In [3]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, Dict

def calculate_liquidity_below_range(x: float, pa: float, pb: float) -> float:
    """
    Calculate liquidity L for a single-sided ETH position (full range below current price).
    Formula: L = x / (1/sqrt(pa) - 1/sqrt(pb))
    """
    sqrt_pa = np.sqrt(pa)
    sqrt_pb = np.sqrt(pb)
    denominator = 1 / sqrt_pa - 1 / sqrt_pb
    if denominator <= 0:
        raise ValueError("Invalid range: pa must be < pb")
    return x / denominator

def calculate_liquidity_above_range(y: float, pa: float, pb: float) -> float:
    """
    Calculate liquidity L for a single-sided USDC position (full range above current price).
    Formula: L = y / (sqrt(pb) - sqrt(pa))
    """
    sqrt_pa = np.sqrt(pa)
    sqrt_pb = np.sqrt(pb)
    denominator = sqrt_pb - sqrt_pa
    if denominator <= 0:
        raise ValueError("Invalid range: pa must be < pb")
    return y / denominator

def position_value_at_price(l: float, pa: float, pb: float, p: float) -> float:
    """
    Calculate the position's value in USDC equivalent at price p.
    - If p <= pa: All USDC, value = L * (sqrt(pb) - sqrt(pa))
    - If p >= pb: All ETH, value = [L * (1/sqrt(pa) - 1/sqrt(pb))] * p
    - If pa < p < pb: Mixed, value = x * p + y where x = L * (1/sqrt(p) - 1/sqrt(pb)), y = L * (sqrt(p) - sqrt(pa))
    """
    sqrt_pa = np.sqrt(pa)
    sqrt_pb = np.sqrt(pb)
    sqrt_p = np.sqrt(p)
    
    if p <= pa:
        # Fully converted to USDC
        return l * (sqrt_pb - sqrt_pa)
    elif p >= pb:
        # Fully converted to ETH
        x_full = l * (1 / sqrt_pa - 1 / sqrt_pb)
        return x_full * p
    else:
        # In-range: partial ETH and USDC
        x = l * (1 / sqrt_p - 1 / sqrt_pb)
        y = l * (sqrt_p - sqrt_pa)
        return x * p + y

def leg1_details_and_pnl(
    capital: float,
    eth_price: float,
    pa: float,
    pb: float,
    leverage: float,
    borrow_apr_usdc: float = 0.10,
    days: float = 0,
    price_ticks: np.ndarray = None
) -> Tuple[float, float, Dict[float, float]]:
    """
    Compute details and PnL for Leg 1 (Bullish Short Put: ETH-heavy position below current price).
    
    Returns:
    - l: Liquidity provided
    - total_eth: Total ETH provided to LP
    - pnls: Dict of {price: pnl} absolute PnL on capital at each price
    """
    if leverage < 1:
        raise ValueError("Leverage must be >= 1")
    
    # Borrow USDC and convert to ETH
    borrow_usdc = capital * (leverage - 1)
    total_eth = (capital + borrow_usdc) / eth_price
    l = calculate_liquidity_below_range(total_eth, pa, pb)
    
    # Verify initial value matches total input
    initial_value = position_value_at_price(l, pa, pb, eth_price)
    assert np.isclose(initial_value, capital * leverage, rtol=1e-6)
    
    # Borrow cost (simple interest)
    debt_cost = borrow_usdc * (borrow_apr_usdc * days / 365)
    
    # Simulate PnL at price ticks
    if price_ticks is None:
        price_ticks = np.linspace(eth_price * 0.6, eth_price * 1.4, 21)  # ~200 USD ticks around current
    
    pnls = {}
    for p in price_ticks:
        pos_value = position_value_at_price(l, pa, pb, p)
        net_value = pos_value - borrow_usdc - debt_cost
        pnl = net_value - capital  # Absolute PnL
        pnls[p] = pnl
    
    return l, total_eth, pnls

def leg2_details_and_pnl(
    capital: float,
    eth_price: float,
    pa: float,
    pb: float,
    leverage: float,
    borrow_apr_eth: float = 0.15,
    days: float = 0,
    price_ticks: np.ndarray = None
) -> Tuple[float, float, Dict[float, float]]:
    """
    Compute details and PnL for Leg 2 (Bearish Short Call: USDC-heavy position above current price).
    
    Returns:
    - l: Liquidity provided
    - total_usdc: Total USDC provided to LP
    - pnls: Dict of {price: pnl} absolute PnL on capital at each price
    """
    if leverage < 1:
        raise ValueError("Leverage must be >= 1")
    
    # Borrow ETH and sell for USDC
    borrow_eth = (capital * (leverage - 1)) / eth_price
    total_usdc = capital + (borrow_eth * eth_price)
    l = calculate_liquidity_above_range(total_usdc, pa, pb)
    
    # Verify initial value matches total input
    initial_value = position_value_at_price(l, pa, pb, eth_price)
    assert np.isclose(initial_value, capital * leverage, rtol=1e-6)
    
    # Borrow cost (simple interest on initial debt value)
    initial_debt_value = borrow_eth * eth_price
    debt_cost = initial_debt_value * (borrow_apr_eth * days / 365)
    
    # Simulate PnL at price ticks
    if price_ticks is None:
        price_ticks = np.linspace(eth_price * 0.6, eth_price * 1.4, 21)
    
    pnls = {}
    for p in price_ticks:
        pos_value = position_value_at_price(l, pa, pb, p)
        debt_value = borrow_eth * p
        net_value = pos_value - debt_value - debt_cost
        pnl = net_value - capital
        pnls[p] = pnl
    
    return l, total_usdc, pnls

def combined_pnl(leg1_pnls: Dict[float, float], leg2_pnls: Dict[float, float]) -> Dict[float, float]:
    """Combine PnL from both legs."""
    prices = sorted(leg1_pnls.keys())
    combined = {p: leg1_pnls[p] + leg2_pnls[p] for p in prices}
    return combined

def plot_pnl_curves(leg1_pnls: Dict[float, float], leg2_pnls: Dict[float, float], combined_pnls: Dict[float, float]):
    """Plot PnL curves vs. ETH price."""
    prices = sorted(leg1_pnls.keys())
    plt.figure(figsize=(12, 8))
    plt.plot(prices, [leg1_pnls[p] for p in prices], label='Leg 1 (Bullish Short Put)', linewidth=2)
    plt.plot(prices, [leg2_pnls[p] for p in prices], label='Leg 2 (Bearish Short Call)', linewidth=2)
    plt.plot(prices, [combined_pnls[p] for p in prices], label='Combined (Delta-Neutral Strangle)', linewidth=2)
    plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    plt.axvline(x=5000, color='red', linestyle='--', alpha=0.5, label='Entry Price ($5,000)')
    plt.xlabel('ETH Price (USD)')
    plt.ylabel('Absolute PnL (USD on $10k Capital)')
    plt.title('Hedged Strangle Strategy: PnL vs. ETH Price Movements')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Example Usage: Run the Strategy
if __name__ == "__main__":
    # Inputs (customizable)
    capital_per_leg = 5000.0  # USD per leg ($10k total)
    eth_price = 5000.0  # Current ETH price
    leverage = 3.0  # 3x leverage
    pa1, pb1 = 4900.0, 5100.0  # Leg 1 range (below)
    pa2, pb2 = 4900.0, 5100.0  # Leg 2 range (above)
    borrow_apr_usdc = 0.10  # 10% APR for USDC borrow (Leg 1)
    borrow_apr_eth = 0.15   # 15% APR for ETH borrow (Leg 2)
    days = 0.0  # Days held (0 for instantaneous ticks; increase for borrow costs)
    
    # Custom price ticks for simulation (every ~200 USD)
    price_ticks = np.arange(3000, 7200, 200)
    
    # Compute Leg 1
    print("=== LEG 1: Bullish Short Put (ETH-Heavy, Range [{}-{}]) ===".format(pa1, pb1))
    l1, total_eth1, pnls1 = leg1_details_and_pnl(
        capital_per_leg, eth_price, pa1, pb1, leverage,
        borrow_apr_usdc, days, price_ticks
    )
    print(f"Liquidity (L): {l1:.2f}")
    print(f"Total ETH Provided: {total_eth1:.4f} ETH")
    print("\nPnL at Price Ticks (Absolute $ on ${:.0f} Capital):".format(capital_per_leg))
    print("ETH Price | PnL ($)")
    print("-" * 20)
    for p in sorted(pnls1.keys()):
        print(f"${p:,.0f}    | ${pnls1[p]:,.2f}")
    
    # Compute Leg 2
    print("\n=== LEG 2: Bearish Short Call (USDC-Heavy, Range [{}-{}]) ===".format(pa2, pb2))
    l2, total_usdc2, pnls2 = leg2_details_and_pnl(
        capital_per_leg, eth_price, pa2, pb2, leverage,
        borrow_apr_eth, days, price_ticks
    )
    print(f"Liquidity (L): {l2:.2f}")
    print(f"Total USDC Provided: ${total_usdc2:,.2f}")
    print("\nPnL at Price Ticks (Absolute $ on ${:.0f} Capital):".format(capital_per_leg))
    print("ETH Price | PnL ($)")
    print("-" * 20)
    for p in sorted(pnls2.keys()):
        print(f"${p:,.0f}    | ${pnls2[p]:,.2f}")
    
    # Combined
    print("\n=== COMBINED: Delta-Neutral Strangle ($10k Total Capital) ===")
    combined_pnls = combined_pnl(pnls1, pnls2)
    print("PnL at Price Ticks (Absolute $ on $10k Capital):")
    print("ETH Price | PnL ($)")
    print("-" * 20)
    for p in sorted(combined_pnls.keys()):
        print(f"${p:,.0f}    | ${combined_pnls[p]:,.2f}")
    
    # Plot (uncomment to visualize; requires matplotlib backend)
    plot_pnl_curves(pnls1, pnls2, combined_pnls)
    print("\nTo visualize: Run plot_pnl_curves(pnls1, pnls2, combined_pnls)")

=== LEG 1: Bullish Short Put (ETH-Heavy, Range [4900.0-5100.0]) ===


AssertionError: 